# RealMLP - GPU run on Kaggle

Runs **[RealMLP](https://github.com/dholzmueller/pytabkit)** (Holzmuller et al., 2024 -
"Better by Default": a strong *pre-tuned* MLP) on the **full** churn training set, logging
a comparable run next to TabM via the shared experiment harness. RealMLP is a
gradient-trained MLP with **no in-context row cap**, so unlike TabPFN / TabICL / Mitra it
runs on all 594k rows. It needs scaled inputs, but `pytabkit` preprocesses internally, so
the one-hot matrix is fed as-is.

**Why Kaggle:** GPU training; the local environment is CPU-only.
**Settings (right sidebar):** Accelerator -> GPU T4 x2 (not P100); Internet -> On;
Add Input -> playground-series-s6e3.

> WARNING: This scaffold mirrors the committed TabM run, but the API calls are **not
> verified against the live environment**. Confirm `RealMLP_TD_Classifier`'s kwargs
> against the installed `pytabkit`, and **smoke-test one fold** (`n_splits=2`) first.

In [1]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)         # makes `from src.xxx import ...` resolve
print("CWD:", os.getcwd())

Cloning into '/kaggle/working/Predict-Customer-Churn'...


CWD: /kaggle/working/Predict-Customer-Churn


In [3]:
!pip install -q pytabkit

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())     # expect 2 on T4 x2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 79.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

In [4]:
# cuda.is_available() can return True on an incompatible GPU - run a real op.
import torch
print("device:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())
try:
    _ = (torch.randn(16, device="cuda") @ torch.randn(16, 16, device="cuda")).sum().item()
    print("GPU compute OK")
except Exception as e:
    print("GPU compute FAILED:", e)            # if this fails, switch to T4 and restart

from pytabkit import RealMLP_TD_Classifier
print("RealMLP_TD_Classifier imported OK")

device: Tesla T4 | count: 2
GPU compute OK
RealMLP_TD_Classifier imported OK


In [5]:
# data/processed/*.parquet are git-ignored, so absent from the clone. Rebuild them
# from the attached competition CSVs.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(force=True)   # writes data/processed/*.parquet

Preprocessed and saved: train_df (594194, 42), test_df (254655, 41)


In [6]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score

from src.tracking import RUNS_DIR
from src.cv import run_cv_experiment, save_experiment

### Design matrix (full data)

`prepare_data` returns the one-hot-encoded frames used across the project. RealMLP handles
the full 594k-row training set (no in-context row cap), so - like TabM - we pass everything
to the CV harness; `pytabkit` scales inputs internally.

In [7]:
encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}  features: {len(encoded_features)}')

X_train: (594194, 40)  X_test: (254655, 40)  features: 40


### Run configuration - RealMLP baseline

Same `run_config` shape as the TabM run. `metric=accuracy_score`; the harness *always*
logs **OOF ROC-AUC** separately (the project's primary metric), so that is the score we
read. `save_models=False` because RealMLP is torch-backed (fragile to `joblib.dump`).

In [8]:
from pytabkit import RealMLP_TD_Classifier

realmlp_params = {
    'device':       'cuda',
    'random_state': 42,
    'n_cv':         1,    # no internal CV ensembling - one model per outer fold
    'n_refit':      0,
}

DATA_VERSION = 'fe_v0'   # identity FE (no engineered features) - baseline

run_config = {
    'model_factory': lambda params: RealMLP_TD_Classifier(**params),
    'params':        realmlp_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'realmlp-baseline',
    'notes': (
        'RealMLP (pytabkit RealMLP_TD_Classifier, tuned defaults) on Kaggle T4 GPU, '
        'full 594k training set. OOF ROC-AUC vs untuned LGBM ~0.91 / tuned ~0.92. '
        'pytabkit=1.7.3, torch=2.10.0+cu128. Data regenerated on-platform - data_hash differs '
        'from local runs; GPU run not bit-reproducible.'
    ),
    'parent_run_id': '',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

In [9]:
# Step 1 - Run the experiment. SMOKE-TEST FIRST: set n_splits=2 in the run_config cell,
# confirm the GPU path works and check per-fold time, before a full Save & Run All.
result = run_cv_experiment(run_config, X_train, y_train, X_test, encoded_features)

Run ID: 20260603-021557-d716c1
Tag:    realmlp-baseline



/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
/usr/lo

Fold 0: accuracy=0.8578  roc_auc=0.9131  (fit 2918.2s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
/usr/lo

Fold 1: accuracy=0.8588  roc_auc=0.9146  (fit 2887.5s)


...

Fold 2: accuracy=0.8589  roc_auc=0.9136  (fit 2860.8s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
/usr/lo

Fold 3: accuracy=0.8597  roc_auc=0.9150  (fit 2874.9s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
/usr/lo

Fold 4: accuracy=0.8583  roc_auc=0.9116  (fit 2867.2s)

OOF accuracy: 0.8587
OOF ROC-AUC:  0.9134
Folds:        0.8587 ± 0.0006

Run complete. Call save_experiment(result) to log this run permanently.


In [10]:
# Step 2 - Save the run (optional). Review the OOF ROC-AUC printed above first.
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260603-021557-d716c1


### Build a submission (optional)

`test_proba_mean` is the fold-bagged churn probability for the full test set. The
competition metric is ROC-AUC, so submit the probability directly.

In [11]:
submission = pd.DataFrame({
    'id':    test_df['id'],
    'Churn': result['artifacts']['test_proba_mean'],
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print('wrote /kaggle/working/submission.csv', submission.shape)

       id     Churn
0  594194  0.109570
1  594195  0.051675
2  594196  0.138362
3  594197  0.053173
4  594198  0.470120
wrote /kaggle/working/submission.csv (254655, 2)


### Artifact hand-off

To fold this run back into the local repo — extract the run dir, append the new
`runs.csv` row, optionally submit and backfill `lb_public` / `lb_private`, and commit
this notebook under `kaggle/` — follow **§7-8 of `docs/kaggle_gpu_workflow.md`**.
Start by zipping the run directory for download (Output tab):

    import shutil
    from src.tracking import RUNS_DIR
    shutil.make_archive(f"/kaggle/working/{run_id}", "zip", RUNS_DIR / run_id)

In [12]:
    import shutil
    from src.tracking import RUNS_DIR
    shutil.make_archive(f"/kaggle/working/{run_id}", "zip", RUNS_DIR / run_id)

'/kaggle/working/20260603-021557-d716c1.zip'